# 02 · Vorverarbeitung

Rohdaten normalisieren, filtern und mit vier zyklischen Prompts speichern.


In [1]:
# Projekt-Wurzel finden und Importe ermöglichen
import pathlib
import sys
from IPython.display import display

PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Projekt:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

Projekt: <Projektordner>/analysis
Python: 3.12.13


## Konfiguration


In [2]:
from pprint import pprint
from src.utils.config import load_yaml, named_config

print("Datensätze:")
for alias in ("medqa", "medmcqa", "mmlu"):
    pprint(named_config("datasets", alias))

print("\nVorverarbeitung:")
pprint(load_yaml("configs/preprocessing/thesis.yaml"))

print("\nPrompt:")
pprint(load_yaml("configs/experiments/thesis.yaml")["prompt"])

Datensätze:
{'adapter': 'medqa',
 'alias': 'medqa',
 'config': None,
 'default_split': 'test',
 'repo': 'openlifescienceai/MedQA-USMLE-4-options-hf',
 'revision': 'main'}
{'adapter': 'medmcqa',
 'alias': 'medmcqa',
 'config': None,
 'default_split': 'validation',
 'repo': 'openlifescienceai/medmcqa',
 'revision': 'main'}
{'adapter': 'mmlu',
 'alias': 'mmlu',
 'config': 'all',
 'default_split': 'test',
 'repo': 'cais/mmlu',
 'revision': 'main'}

Vorverarbeitung:
{'duplicate_sample_ids': 'quarantine_later',
 'exact_duplicate_samples': 'keep_and_report',
 'exclude_duplicate_option_texts': True,
 'exclude_option_label_combinations': True,
 'exclude_relative_position_references': True,
 'normalization_errors': 'quarantine',
 'selection_stage': 'after_filtering',
 'version': 'mcq-preprocessing-v1'}

Prompt:
{'answer_heading': 'Answer:',
 'instruction': 'Select the single best answer. Respond with exactly one '
                'option ID: A, B, C, or D.',
 'question_heading': 'Question:',
 'v

## Normalisierung

Gültig sind Fragen mit vier nicht leeren Optionen und einem Goldindex von 0–3.


In [3]:
from src.data.registry import normalize_row

raw_mmlu = {
    "question": "Which structure pumps blood through the body?",
    "choices": ["Liver", "Heart", "Kidney", "Lung"],
    "answer": 1,
    "subject": "anatomy",
}

sample = normalize_row("mmlu", raw_mmlu, row_index=7)
print(sample)
print("Goldlabel:", sample.gold_label)
print("Serialisierbar:", sample.to_dict())

MCQSample(sample_id='mmlu:anatomy:000007', dataset='mmlu', question='Which structure pumps blood through the body?', options=('Liver', 'Heart', 'Kidney', 'Lung'), gold_index=1, subject='anatomy')
Goldlabel: B
Serialisierbar: {'sample_id': 'mmlu:anatomy:000007', 'dataset': 'mmlu', 'question': 'Which structure pumps blood through the body?', 'options': ['Liver', 'Heart', 'Kidney', 'Lung'], 'gold_index': 1, 'subject': 'anatomy', 'gold_label': 'B'}


## Filter

Positionsverweise (`none of the above`), Meta-Antworten (`Both A and C`) und doppelte Optionstexte werden ausgeschlossen. Fachliche Buchstabenangaben bleiben erhalten.


In [4]:
import pandas as pd
from src.data.preprocessing import (
    duplicate_option_texts,
    has_display_label_reference,
    has_option_label_combination,
    has_relative_position_reference,
)

examples = [
    "None of the above",
    "Both A and C",
    "Vitamin A and D",
    "Countries A and B have equal income",
]

pd.DataFrame({
    "Text": examples,
    "relative Position": [has_relative_position_reference(x) for x in examples],
    "reine A-D-Kombination": [has_option_label_combination(x) for x in examples],
})

,Text,relative Position,reine A-D-Kombination
0,None of the above,True,False
1,Both A and C,False,True
2,Vitamin A and D,False,False
3,Countries A and B have equal income,False,False


Bei mindestens drei Optionen mit Buchstabenkombinationen oder Reihenfolgen behandelt der Filter diese als Aufgabeninhalt.


In [5]:
option_sets = {
    "unsichere Meta-Antwort": ["one", "two", "three", "Both A and C"],
    "fachliche Sequenzaufgabe": ["A,B,C,D", "A,C,D,B", "C,A,D,B", "D,C,B,A"],
    "doppelte Optionen": ["same", "same", "three", "four"],
}

for name, options in option_sets.items():
    print(
        f"{name:27s}",
        "ID-Verweis=", has_display_label_reference(options),
        "Duplikat=", duplicate_option_texts(options),
    )

unsichere Meta-Antwort      ID-Verweis= True Duplikat= False
fachliche Sequenzaufgabe    ID-Verweis= False Duplikat= False
doppelte Optionen           ID-Verweis= False Duplikat= True


## Ausschlüsse prüfen

`preprocess_rows` liefert akzeptierte Samples, Quarantäne und Bericht. Pro Zeile sind mehrere Ausschlussgründe möglich.


In [6]:
from src.data.preprocessing import PreprocessingPolicy, preprocess_rows

def mmlu_row(question, choices, answer=0):
    return {
        "question": question,
        "choices": choices,
        "answer": answer,
        "subject": "demo",
    }

raw_rows = [
    mmlu_row("usable 1", ["one", "two", "three", "four"]),
    mmlu_row("relative", ["one", "two", "three", "none of the above"]),
    mmlu_row("meta", ["one", "two", "three", "Both A and C"]),
    mmlu_row("duplicate", ["same", "same", "three", "four"]),
    mmlu_row("invalid", ["only", "two"]),
    mmlu_row("usable 2", ["five", "six", "seven", "eight"], answer=1),
]

result = preprocess_rows(
    raw_rows,
    adapter="mmlu",
    normalizer=normalize_row,
    policy=PreprocessingPolicy(),
)

print("Akzeptiert:", [x.question for x in result.samples])
print("Bericht:")
pprint(result.report)
print("\nQuarantäne:")
pd.DataFrame([x.to_dict() for x in result.excluded])[["source_index", "question", "reason_codes", "detail"]]

Akzeptiert: ['usable 1', 'usable 2']
Bericht:
{'accepted_rows_before_selection': 2,
 'accepted_sample_set_sha256': '5ee881dadc20d893ea090593b7e65094859b1f0bffb2ecf0ef62ac573bcc61e3',
 'exact_duplicate_content_extra_rows_retained': 0,
 'exact_duplicate_content_groups_retained': 0,
 'excluded_by_reason': {'duplicate_option_text': 1,
                        'normalization_error': 1,
                        'option_label_combination': 1,
                        'relative_position_reference': 1},
 'excluded_rows': 4,
 'policy': {'duplicate_sample_ids': 'quarantine_later',
            'exact_duplicate_samples': 'keep_and_report',
            'exclude_duplicate_option_texts': True,
            'exclude_option_label_combinations': True,
            'exclude_relative_position_references': True,
            'normalization_errors': 'quarantine',
            'selection_stage': 'after_filtering',
            'version': 'mcq-preprocessing-v1'},
 'policy_sha256': 'd25c5e2fcf6cad91df2e675b9a0674332325

,source_index,question,reason_codes,detail
0,1,relative,[relative_position_reference],NaN
1,2,meta,[option_label_combination],NaN
2,3,duplicate,[duplicate_option_text],NaN
3,4,invalid,[normalization_error],ValueError: MMLU row must contain four choices...


## Stichprobe auswählen

`start_index` und `limit` greifen erst nach der Filterung.


In [7]:
selected = result.select(start_index=1, limit=1)
print("Ausgewählt:", [x.question for x in selected.samples])
print("Sample-Set-Hash:", selected.report["selected_sample_set_sha256"] )

Ausgewählt: ['usable 2']
Sample-Set-Hash: 1d594fb498887b8e2260fb6f11483d511b7d604d739d84c90ee9a3078f1a48e2


## Artefakt speichern

`samples.parquet`, `cyclic_prompts.parquet`, `excluded_rows.parquet` und `manifest.json` enthalten Daten, Zuordnungen, Ausschlüsse und Prüfsummen. Das Beispiel verwendet ein temporäres Verzeichnis.


In [8]:
import json
import tempfile
from src.data.preprocessing import load_preprocessed_artifact, write_preprocessed_artifact
from src.prompts.mcq import MCQPrompt, PromptConfig

prompt = MCQPrompt(PromptConfig("demo-prompt-v1", "Choose exactly one option ID."))

with tempfile.TemporaryDirectory() as tmp:
    artifact = write_preprocessed_artifact(
        result,
        pathlib.Path(tmp) / "demo-artifact",
        prompt_builder=prompt,
        dataset_metadata={"dataset_alias": "mmlu", "dataset_split": "demo"},
    )
    manifest = json.loads((artifact / "manifest.json").read_text())
    samples_df = pd.read_parquet(artifact / "samples.parquet")
    prompts_df = pd.read_parquet(artifact / "cyclic_prompts.parquet")
    loaded_samples, _ = load_preprocessed_artifact(artifact)  # prüft SHA-256

    print("Dateien:", sorted(p.name for p in artifact.iterdir()))
    print("Samples:", len(samples_df), "| Prompts:", len(prompts_df))
    print("Prüfsumme verifiziert:", len(loaded_samples) == len(samples_df))
    print("Policy-Hash:", manifest["policy_sha256"])
    display(prompts_df[["sample_id", "permutation_id", "display_to_content", "gold_displayed_label"]].head(8))

Dateien: ['cyclic_prompts.parquet', 'excluded_rows.parquet', 'manifest.json', 'samples.parquet']
Samples: 2 | Prompts: 8
Prüfsumme verifiziert: True
Policy-Hash: d25c5e2fcf6cad91df2e675b9a0674332325b34e34477812490cc633ced191c3


,sample_id,permutation_id,display_to_content,gold_displayed_label
0,mmlu:demo:000000,0,"[0, 1, 2, 3]",A
1,mmlu:demo:000000,1,"[1, 2, 3, 0]",D
2,mmlu:demo:000000,2,"[2, 3, 0, 1]",C
3,mmlu:demo:000000,3,"[3, 0, 1, 2]",B
4,mmlu:demo:000005,0,"[0, 1, 2, 3]",B
5,mmlu:demo:000005,1,"[1, 2, 3, 0]",A
6,mmlu:demo:000005,2,"[2, 3, 0, 1]",D
7,mmlu:demo:000005,3,"[3, 0, 1, 2]",C


Reale Artefakte erzeugen: siehe [RUN_PREPROCESSING.md](../RUN_PREPROCESSING.md). Anschließend mit [03](03_vorverarbeitung_audit.ipynb) prüfen.
